# One-Click Master Runner & Comparison Dashboard (`MultiEMit — final_codes`)
Click **"Run All"** to execute all Textual (`FEEL-IT` / `UmBERTo`) and Multimodal Dual-Encoder (`+ ViT-224`) experiments across all three tasks on `datasets/corpora/correctly_split/` and compile a **single unified comparison output**:
- `./results/all_experiments_summary.csv` (Master CSV table of all tasks & regimes)
- `./results/all_experiments_summary.xlsx` (Multi-sheet Excel workbook: `All_Experiments`, `Task1_Emotion`, `Task2_Irony`, `Task3_Relation`)
- `./results/all_experiments_full_report.md` (Full Markdown report with all tables + every per-label `classification_report`)

### Standard Pipeline Execution Order
1. `train_textual_emotion.ipynb` — Task 1: Textual Emotion Detection (`FEEL-IT`)
2. `train_textual_irony.ipynb` — Task 2: Textual Irony & Sarcasm Detection (`UmBERTo`)
3. `train_textual_relation.ipynb` — Task 3: Textual Semiotic Relation (`UmBERTo`)
4. `multimodal_dual_encoder_emotion.ipynb` — Task 1: Multimodal Emotion Dual-Encoder (`FEEL-IT + ViT-224`)
5. `multimodal_dual_encoder_irony.ipynb` — Task 2: Multimodal Irony Dual-Encoder (`UmBERTo + ViT-224`)
6. `multimodal_dual_encoder_relation.ipynb` — Task 3: Multimodal Semiotic Relation Dual-Encoder (`UmBERTo + ViT-224`)


In [ ]:
import pandas as pd
from IPython.display import Markdown, display
import importlib
import run_all_experiments as runner
importlib.reload(runner)

# ==============================================================================
# ONE-CLICK CONFIGURATION
# ==============================================================================
RUN_TEXTUAL_NOTEBOOKS = True      # Set False if textual checkpoints are already trained and you only want to run Multimodal
RUN_MULTIMODAL_NOTEBOOKS = True   # Set True to run all 3 Multimodal Dual-Encoder (+ ViT-224) notebooks
ONLY_AGGREGATE_EXISTING = False   # Set True to skip training/inference and immediately build the unified comparison tables
FORCE_RETRAIN_OVERRIDE = True     # True = retrain all checkpoints from scratch; False = reuse existing ./checkpoints/* if present


In [ ]:
importlib.reload(runner)

if not ONLY_AGGREGATE_EXISTING:
    pipeline_to_run = []
    if RUN_TEXTUAL_NOTEBOOKS:
        pipeline_to_run.extend(runner.TEXTUAL_NOTEBOOKS)
    if RUN_MULTIMODAL_NOTEBOOKS:
        pipeline_to_run.extend(runner.MULTIMODAL_NOTEBOOKS)

    execution_records = []
    for nb_file, label in pipeline_to_run:
        rec = runner.run_notebook(nb_file, label, force_retrain_override=FORCE_RETRAIN_OVERRIDE)
        execution_records.append(rec)

    print("\n=== Pipeline Execution Status ===")
    display(pd.DataFrame(execution_records))


## Unified Comparison Table (All Tasks, Modalities & Configurations)

In [ ]:
master_df = runner.aggregate_all_results()
if not master_df.empty:
    print("=== Master Comparison Table (All Experiments) ===")
    display(master_df)

    for task_title, kw in [
        ("Task 1: Emotion Detection (10-Label Multi-Label)", "Emotion"),
        ("Task 2: Irony & Sarcasm Detection (3-Class Single-Label)", "Irony"),
        ("Task 3: Semiotic Relation Classification (3-Class Single-Label)", "Relation"),
    ]:
        sub_df = master_df[master_df["Task"].str.contains(kw, case=False, na=False)].drop(columns=["Task"])
        if not sub_df.empty:
            print(f"\n=== {task_title} ===")
            display(sub_df.reset_index(drop=True))
